## Importing Dependencies

In [1]:
# imports
import os
from pathlib import Path
from tqdm import tqdm
import cv2
import filetype
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
from PIL import Image
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Path Setup & Class Detection

In [2]:
# setting up source & output paths
SRC_DIR = Path("data/original_images")
OUT_DIR = Path("data/split")  
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# checking for classes 
CLASS_NAMES = sorted([p.name for p in SRC_DIR.iterdir() if p.is_dir()])
print("Detected classes:", CLASS_NAMES)

Detected classes: ['binder', 'files', 'headphone', 'monitor', 'mug', 'printer']


## Verifying & Collecting Images

In [4]:
# valid file extensions
image_extensions = ['jpeg', 'jpg', 'bmp', 'png']

rows = []

for cls in CLASS_NAMES:
    folder = SRC_DIR / cls
    print(f"Checking {cls} ...")
    
    for image_file in tqdm(os.listdir(folder)):
        image_path = folder / image_file

        if not image_path.is_file():
            continue  
        try:
            # Read image
            img = cv2.imread(str(image_path))
            kind = filetype.guess(str(image_path))

            # Validate using cv2 and filetype
            if img is None or kind is None or kind.extension not in image_extensions:
                print(f"Invalid or unreadable image: {image_path}")
                os.remove(image_path)
                continue

            # Check if image is RGB
            if len(img.shape) == 2:  
                # grayscale (H, W)
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
                cv2.imwrite(str(image_path), img)
                print(f"Converted grayscale to RGB: {image_path}")
            elif img.shape[2] == 4:  
                # RGBA (H, W, 4)
                img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
                cv2.imwrite(str(image_path), img)
                print(f"Removed alpha channel (converted RGBA to RGB): {image_path}")

            # Store valid image info
            rows.append({"filepath": str(image_path), "label": cls})

        except Exception as e:
            print(f"Error reading image {image_path}: {e}")
            os.remove(image_path)

Checking binder ...


 20%|██        | 308/1519 [00:00<00:00, 3074.10it/s]

Invalid or unreadable image: data/original_images/binder/.DS_Store


100%|██████████| 1519/1519 [00:00<00:00, 3498.25it/s]


Checking files ...


...

100%|██████████| 1559/1559 [00:00<00:00, 5303.39it/s]


Checking printer ...


 25%|██▍       | 380/1546 [00:00<00:00, 3797.03it/s]

Invalid or unreadable image: data/original_images/printer/.DS_Store


100%|██████████| 1546/1546 [00:00<00:00, 3694.37it/s]


In [5]:
# create dataframe
df_office = pd.DataFrame(rows)
print(f"\nTotal verified images: {len(df_office)}")


Total verified images: 8740


In [6]:
# print dataframe
df_office

,filepath,label
0,data/original_images/binder/GI3P1HZ6LQUZ.jpg,binder
1,data/original_images/binder/6E3Z4YGGPPKH.jpg,binder
2,data/original_images/binder/ZM8UL9HF2JC6.jpg,binder
3,data/original_images/binder/OJXR0F2CYWUU.jpg,binder
4,data/original_images/binder/4H15VIGAD7Q4.jpg,binder
...,...,...
8735,data/original_images/printer/VODDHFN4IXK4.jpg,printer
8736,data/original_images/printer/8BCYHKALVF6C_aug.jpg,printer
8737,data/original_images/printer/RBUPD7KLC5QT_aug.jpg,printer
8738,data/original_images/printer/YQV25WH5623A_aug.jpg,printer


## Save CSV with all images & labels

In [7]:
csv_path = "data/office_items.csv"
df_office.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path}")

Saved CSV: data/office_items.csv


## Data Preprocessing & Analysis

In [8]:
# distribution of classes
df_office['label'].value_counts()

label
mug          1558
printer      1545
binder       1518
monitor      1510
files        1437
headphone    1172
Name: count, dtype: int64

In [9]:
# number of rows & columns
df_office.shape

(8740, 2)

In [10]:
# check for missing values
df_office.isnull().sum()

filepath    0
label       0
dtype: int64

In [11]:
# drop duplicate values
df_office.drop_duplicates(subset=['filepath'], inplace=True)

In [12]:
# File extensions in dataset
print(df_office['filepath'].apply(lambda x: Path(x).suffix.lower()).value_counts())

filepath
.jpg     8410
.jpeg     200
.png      130
Name: count, dtype: int64


## Train, Val & Test split

In [13]:
trainval, test = train_test_split(df_office, test_size=0.10, stratify=df_office['label'], random_state=99)
train, val = train_test_split(trainval, test_size=0.111111, stratify=trainval['label'], random_state=99)

In [14]:
print(f"Split counts -> Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Split counts -> Train: 6992, Val: 874, Test: 874


## Copy images to ImageFolder layout

In [15]:
def copy_to_split(split_df, split_name):
    missing_files = []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Copying {split_name}"):
        src = Path(row['filepath'])
        if not src.exists():
            missing_files.append(src)
            continue  # skip missing file
        dst_dir = OUT_DIR / split_name / row['label']
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        if not dst.exists():
            shutil.copy(src, dst)
    if missing_files:
        print(f"Skipped {len(missing_files)} missing files:")
        for f in missing_files:
            print(f"  {f}")

copy_to_split(train, "train")
copy_to_split(val, "val")
copy_to_split(test, "test")


Copying test: 100%|██████████| 874/874 [00:00<00:00, 3953.39it/s]


## Dataset mean & std

In [16]:
def compute_mean_std(df_subset):
    means, stds = [], []
    for p in tqdm(df_subset['filepath'], desc="Computing mean/std"):
        img = Image.open(p).convert("RGB")
        arr = np.asarray(img, dtype=np.float32) / 255.0
        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))
    mean = np.mean(means, axis=0)
    std = np.mean(stds, axis=0)
    return mean, std

mean, std = compute_mean_std(train.sample(200))
print("Mean:", mean)
print("Std :", std)

Computing mean/std: 100%|██████████| 200/200 [00:01<00:00, 185.01it/s]

Mean: [0.47221252 0.44880754 0.43034497]
Std : [0.20280607 0.20500153 0.20543486]


## Load datasets using Keras

In [17]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE

# Load datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    OUT_DIR / "train",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    color_mode="rgb" 
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    OUT_DIR / "val",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    color_mode="rgb" 
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    OUT_DIR / "test",
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    color_mode="rgb" 
)

Found 6992 files belonging to 6 classes.
Found 874 files belonging to 6 classes.
Found 874 files belonging to 6 classes.


In [18]:
# Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.3),
    layers.RandomBrightness(factor=0.2)
], name="data_augmentation")

# Normalization layer
normalization_layer = layers.Rescaling(1./255)

# Apply augmentation first, then normalization, then caching/prefetch
train_ds_augmented = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
).map(
    lambda x, y: (normalization_layer(x), y)
).cache().shuffle(1000).prefetch(AUTOTUNE)

# Validation and test datasets only need normalization
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)
